In [18]:
#cell 0
from pathlib import Path
import os
import sys

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / "requirements.txt").exists() and (p / "api").exists():
            return p
    return cur.resolve()

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
print("CWD =", os.getcwd())


CWD = D:\logistics_AI


In [19]:
#cell 1
from __future__ import annotations
from pathlib import Path
import sys
import pandas as pd
import numpy as np

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / "requirements.txt").exists() and (p / "api").exists():
            return p
    return cur.resolve()

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from api.process_ai.registry_loader import load_registry, get_step_codes
from api.process_ai.validate import validate_events_df
from api.process_ai.features import build_case_feature_matrix, compute_baselines


In [20]:
# Cell 2
import pandas as pd
from api.process_ai.registry_loader import load_registry, get_step_codes
from api.process_ai.validate import validate_events_df
from api.process_ai.features import build_case_feature_matrix

PROCESS_CODES = [
    "TRUCKING_DELIVERY_FLOW",
    "WAREHOUSE_FULFILLMENT",
    "IMPORT_CUSTOMS_CLEARANCE"
]
SEGMENT_COL = "distance_bucket"   # trucking
# warehouse có thể dùng: warehouse_code
# customs có thể dùng: port_code

DATA_DIR = "data/synth_optimal_3process_v1"
REGISTRY_DIR = "registry/synth_optimal_v1"

events = pd.read_csv(f"{DATA_DIR}/events.csv")
ctx = pd.read_csv(f"{DATA_DIR}/cases_context.csv")

# chỉ lấy đúng process đang phân tích
events = events[events["process_code"] == PROCESS_CODE].copy()
ctx = ctx.copy()   # cases_context của bộ synth không có process_code

reg = load_registry(REGISTRY_DIR, PROCESS_CODE)
step_codes = get_step_codes(reg)

events, rep = validate_events_df(events, PROCESS_CODE, step_codes, allow_unknown_steps=True)

feat, schema, fr = build_case_feature_matrix(
    events,
    step_codes,
    cases_context_df=ctx,
    include_context_numeric=True
)

feat.shape, ctx[SEGMENT_COL].value_counts().head()

((12000, 41),
 distance_bucket
 2    4190
 1    4142
 3    2486
 4    1182
 Name: count, dtype: int64)

In [21]:
# Cell 3

# Global baseline: dùng toàn bộ cases
base_global = compute_baselines(feat, step_codes, normal_case_ids=None)

# Segment baselines
seg_baselines = {}
for seg_value, sub in ctx.groupby(SEGMENT_COL):
    sub_ids = sub["case_id"].astype(str).tolist()

    # ít quá thì baseline không ổn
    if len(sub_ids) < 200:
        continue

    seg_baselines[seg_value] = compute_baselines(
        feat,
        step_codes,
        normal_case_ids=sub_ids
    )

len(seg_baselines), list(seg_baselines.keys())[:5]



(4, [1, 2, 3, 4])

In [22]:
#cell 4
# pick 1 case risk cao (ví dụ)
case_id = feat.index[0]
seg_value = ctx.set_index("case_id").loc[case_id, SEGMENT_COL]

def top_deviation(case_id: str, baselines):
    row = feat.loc[case_id]
    best = None
    for s in step_codes:
        dur = float(row[f"{s}_duration_min"])
        p95 = float(baselines["steps"][s]["p95"])
        dev = (dur/p95) if p95>0 else 0.0
        if best is None or dev > best[0]:
            best = (dev, s, dur, p95)
    return best

g = top_deviation(case_id, base_global)
s = top_deviation(case_id, seg_baselines.get(seg_value, base_global))

print("Case:", case_id, "segment:", seg_value)
print("Global top:", g)
print("Segment top:", s)


Case: ORD_0000312750 segment: 1
Global top: (1.4889187374076562, 'STEP_025_DELIVERED_CONFIRMED', 36.95, 24.816666666666666)
Segment top: (1.4879693949461394, 'STEP_025_DELIVERED_CONFIRMED', 36.95, 24.832499999999996)
